In [41]:
import pandas as pd
import json
import os
import matplotlib as mpl
from matplotlib import pyplot as plt
import requests
from io import StringIO as sio
from matplotlib.patches import Patch
import matplotlib.ticker as ticker
import re
import itertools
import sys
import baltic as bt
import random
from collections import Counter
import matplotlib.pyplot as plt
from scipy.stats import norm
import numpy as np
import seaborn as sns

**This code computes pairwise divergence values between sequences in the avian NA alignment to generate divergence cutoffs for within and between subtype reassortments:**

In [42]:
def fasta_to_df(fasta_file):
    
    fasta_data = []
    
    with open(fasta_file) as f:
        header = ""
        sequence = ""
        for line in f:
            if line.startswith(">"):
                if header != "":
                    fasta_data.append({"header": header, "sequence": sequence})
                header = line.strip() 
                sequence = ""
            else:
                sequence += line.strip()
        fasta_data.append({"header": header, "sequence": sequence}) #last line
            
    return pd.DataFrame(fasta_data)

def fasta_writer(path, filename, df):
            
    try:  
        os.mkdir(path)

    except OSError as error:
        pass

    with open(f"{path}{filename}", "w") as f:
        for index, row in df.iterrows():
            f.write(f"{row['header']}\n")
            f.write(f"{row['sequence']}\n")


In [43]:
def compute_divergence(seq1, seq2):
    
    s1 = np.frombuffer(seq1.encode("utf-8"), dtype='S1')
    s2 = np.frombuffer(seq2.encode("utf-8"), dtype='S1')

    valid = (
        (s1 != b'-') & (s2 != b'-') &
        (s1 != b'N') & (s2 != b'N') &
        (s1 != b'n') & (s2 != b'n')
    )

    n_valid = valid.sum()
        
    if n_valid == 0:
        return np.nan
    
    mismatches = np.sum((s1 != s2) & valid) # not dividing by sites to keep consistent with treesort div values
    return mismatches 

df2 = fasta_to_df("files/h3nx_na.fasta")
meta = pd.read_csv("files/metadata.csv")

df2["strain"] = df2["header"].str.lstrip(">")
df2 = df2.merge(meta, left_on="strain", right_on="strain", how="left")

# mistyped strains, need to fix in the metadata as well 
exclude_strains = [
    "A/American_black_duck/Newfoundland/734/2008|2008-09-19",
    "A/mallard/Ohio/184/1986|1986-11-06",
    "A/blue-winged_teal/Guatemala/CIP049H102-32/2011|2011-11-11"
]

df2 = df2[~df2["strain"].isin(exclude_strains)]

# filtering out low quality sequences

df2["seq_len"] = df2["sequence"].str.len()

def count_valid_sites(seq):
    s = np.frombuffer(seq.encode("utf-8"), dtype='S1')
    valid = (s != b'-') & (s != b'N') & (s != b'n')
    return valid.sum()

df2["n_valid_sites"] = df2["sequence"].apply(count_valid_sites)

# maximum length per subtypes 

max_len_per_subtype = df2.groupby("subtype")["seq_len"].max().to_dict()

length_threshold = 0.8   # keep sequences ≥80% of max subtype length
valid_threshold = 0.8    # keep sequences with ≥80% valid sites relative to their length

df_filtered = df2[
    df2.apply(
        lambda row: (row.seq_len >= length_threshold * max_len_per_subtype[row.subtype]) and
                    (row.n_valid_sites / row.seq_len >= valid_threshold),
        axis=1
    )
]

records = []
for (i, row1), (j, row2) in itertools.combinations(df_filtered.iterrows(), 2):
    div = compute_divergence(row1.sequence, row2.sequence)
    records.append({
        "strain1": row1.strain,
        "strain2": row2.strain,
        "subtype1": row1.subtype,
        "subtype2": row2.subtype,
        "divergence": div,
        "comparison": "within" if row1.subtype == row2.subtype else "between"
    })

div_df = pd.DataFrame(records)


In [44]:
removed_df = df2[~df2["strain"].isin(df_filtered["strain"])]

if removed_df.empty:
    print("No sequences were removed after filtering.")
else:
    removed_counts = removed_df.groupby("subtype").size()
    print("Sequences removed per subtype:")
    for subtype, count in removed_counts.items():
        print(f"{subtype}: {count}")
        
num_pairs = len(div_df)
print(f"Total pairwise comparisons: {num_pairs}")

Sequences removed per subtype:
H3N2: 1
H3N7: 3
H3N8: 1
Total pairwise comparisons: 702705


In [ ]:
def make_pair(s1, s2):
    pair = "_".join(sorted([s1.replace("H3", "").lower(), s2.replace("H3", "").lower()]))
    return pair

div_df["subtype_pair"] = div_df.apply(lambda x: make_pair(x.subtype1, x.subtype2), axis=1)
div_df["comparison"] = div_df.apply(lambda x: "within" if x.subtype1 == x.subtype2 else "between", axis=1)


plt.figure(figsize=(10, 6))
sns.stripplot(
    data=div_df,
    x="subtype_pair",
    y="divergence",
    hue="comparison",
    dodge=False,
    palette={"within": "#0072B2", "between": "black"},
    alpha=0.8,
    size=4
)

plt.xticks(rotation=45, ha="right")
plt.xlabel("subtype pair")
plt.ylabel("pairwise divergence")
sns.despine()
plt.tight_layout()
plt.savefig("plots/rea_events_div_cutoffs.pdf", dpi=300, bbox_inches="tight")
plt.show()


**calcualting within and between subtype divergence cutoffs by taking the max within div value and min between div value:**

In [ ]:
# calculating cuttoffs based on min/max

within_cutoff = div_df.loc[div_df["comparison"]=="within", "divergence"].max()
between_cutoff = div_df.loc[div_df["comparison"]=="between", "divergence"].min()

print(f"Lower cutoff (within): {within_cutoff}")
print(f"Upper cutoff (between): {between_cutoff}")

In [ ]:
plt.figure(figsize=(10, 6))

palette = {"within": "#0072B2", "between": "black"}

sns.histplot(
    data=div_df,
    x="divergence",
    hue="comparison",
    bins=50,
    stat="probability", # scales y-axis to fraction of total comparisons
    element="step",
    palette=palette 
)

plt.xlim(left=0)

# vertical dashed lines for global cutoffs
plt.axvline(within_cutoff, color='#0072B2', linestyle='--', linewidth=2, label='within-subtype cutoff')
plt.axvline(between_cutoff, color='black', linestyle='--', linewidth=2, label='between-subtype cutoff')

plt.xlabel("per-site divergence")
plt.ylabel("fraction of pairwise comparisons")
plt.title("within vs between subtype divergence")
plt.legend().set_visible(False)
plt.grid(alpha=0.3)
plt.savefig("plots/pairwise_div_cutoffs_histo.pdf", dpi=300, bbox_inches="tight")
plt.show()


**now, use these cutoffs to call NA reassortments as between or within subtype:**

In [ ]:
NA_div = {}

with open(f'files/summary.json', "r") as file:
    data = json.load(file)

for name, node_data in data["nodes"].items():
    
    # only consider nodes that list NA as a high support reassorting segment
    
    if "NA" in node_data.get("segments", ""):
        
        divergence_str = node_data.get("divergence", "")
        
        # only extract the divergence value for NA
        
        match = re.search(r"NA\((\d+)\)", divergence_str)
        
        NA_div[name] = int(match.group(1))

In [ ]:
print("there are " + str(len(NA_div)) + " NA reassortments") # how many NA reassortments

In [ ]:
# getting subtype info for each node

with open('files/traits.json', "r") as file:
    data = json.load(file)

traits = {}

for name, node_data in data["nodes"].items():
    traits[name] = {
        "subtype": node_data.get("subtype"),
        "subtype_confidence": node_data.get("subtype_confidence")
    }

In [ ]:
# need to do this to summary tree if you want to load it in with baltic

# with open("files/summary.nwk", 'r') as file:
#     tree = file.read()
    
# # removing commas between segments
# modified = re.sub(r'rea=([^]\)]+)', lambda match: f'rea="{match.group(1).replace(",", "-")}"', tree)

# # removing quotation marks around node names
# modified = re.sub(r"'(NODE_\d+)'", r'\1', modified)

# with open("filessummary_baltic.nwk", "w") as output:
#     output.write(modified.strip())


In [ ]:
tree = bt.loadNewick("files/summary_baltic.nwk", absoluteTime= False)

In [ ]:
count = 0
rea_type = {}

for k in tree.Objects:
    
    if k.traits.get("is_reassorted"):
        
        rea = k.traits.get("rea")
        rea_list = [seg.strip() for seg in rea.split("-")]

        if "NA" in rea_list:
            
            count += 1
            
            name = k.traits["label"] if k.is_node() else k.name
            parent = k.parent.traits["label"] if k.parent.is_node() else k.parent.name
            
            current_sub = traits[name]["subtype"]
            
            parent_sub = traits[parent]["subtype"]
            
            # only looking at reassortments we can confidently call as within or between
            
            if NA_div[name] <= within_cutoff and parent_sub == current_sub:
                rea_type[name] = "within"
                
            elif NA_div[name] >= between_cutoff and parent_sub != current_sub:
                rea_type[name] = "between"

In [ ]:
within_vals = [NA_div[k] for k in NA_div if rea_type.get(k) == "within"]
between_vals = [NA_div[k] for k in NA_div if rea_type.get(k) == "between"]

# shared bin sizes
all_vals = np.array(within_vals + between_vals)
bins = np.linspace(all_vals.min(), all_vals.max(), 21)  # 20 bins → 21 edges

plt.figure(figsize=(10,6))

plt.hist(within_vals, bins=bins, alpha=0.6, label="within", edgecolor="black")
plt.hist(between_vals, bins=bins, alpha=0.6, label="between", edgecolor="black")

plt.xlabel("TreeSort divergence value for reassorting NA")
plt.ylabel("count")
plt.tight_layout()
plt.savefig("plots/within-between_reassortments.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
print(len(within_vals))
print(len(between_vals))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binom

# within
n1 = len(within_vals) + len(between_vals)
successes1 = len(within_vals)
p1 = 0.37
k1 = np.arange(0, n1 + 1)
pmf1 = binom.pmf(k1, n1, p1)

# within
p_value_within = 1- binom.cdf(successes1 - 1, n1, p1) # right-tailed, probability of ≥ observed

plt.figure(figsize=(6,4))
plt.bar(k1, pmf1, alpha=0.4, color='cadetblue')
plt.axvline(successes1, linewidth=3, color='cadetblue')
plt.xlabel("Number of successes")
plt.ylabel("Probability")
plt.title(f"Binomial(n={n1}, prob={p1}), observed={successes1}")
plt.tight_layout()
plt.savefig("plots/binomial_within.pdf", dpi=300, bbox_inches="tight")
plt.show()

# between
n2 = len(within_vals) + len(between_vals)
successes2 = len(between_vals)
p2 = 0.63
k2 = np.arange(0, n2 + 1)
pmf2 = binom.pmf(k2, n2, p2)

# between
p_value_between = binom.cdf(successes2 - 1, n2, p2) # left-tailed, Probability of ≤ observed

plt.figure(figsize=(6,4))
plt.bar(k2, pmf2, alpha=0.4, color='brown')
plt.axvline(successes2, linewidth=3, color='brown')
plt.xlabel("Number of successes")
plt.ylabel("Probability")
plt.title(f"Binomial(n={n2}, prob={p2}), observed={successes2}")
plt.savefig("plots/binomial_between.pdf", dpi=300, bbox_inches="tight")
plt.tight_layout()
plt.show()

**All NA reassortment events, differentiated by whether the reassorting branch is a leaf or node:**

In [ ]:
df = pd.DataFrame(list(NA_div.items()), columns=["name", "divergence"])

# node or a leaf
df["type"] = df["name"].apply(lambda x: "node" if x.startswith("NODE_") else "leaf")
counts = df["type"].value_counts()
print(counts)

In [ ]:
colors = {"leaf": "#5ab4ac", "node": "#d8b365"}

plt.figure(figsize=(10, 6))

plt.hist(
    df.loc[df['type'] == 'node', 'divergence'],
    bins=20,
    alpha=0.6,
    color=colors['node'],
    label='node'
)

plt.hist(
    df.loc[df['type'] == 'leaf', 'divergence'],
    bins=20,
    alpha=0.6,
    color=colors['leaf'],
    label='leaf'
)

# cutoffs calculated above
plt.axvline(within_cutoff, color='#0072B2', linestyle='--', linewidth=2, label='within-subtype')
plt.axvline(between_cutoff, color='black', linestyle='--', linewidth=2, label='between-subtype')

plt.xlim(left=0)
plt.xlabel("TreeSort divergence value for reassorting NA")
plt.ylabel("count")
# plt.legend()
plt.grid(alpha=0.3)
plt.savefig("plots/rea_events_leaf-node_div_cutoffs.pdf", dpi=300, bbox_inches="tight")
plt.show()



In [ ]:
# not separating by leaf/node for presentations

plt.figure(figsize=(10, 6))

plt.hist(
    df['divergence'],
    bins=20,
    alpha=0.8,
    color="#5ab4ac"
)

# cutoffs
plt.axvline(within_cutoff, color='#0072B2', linestyle='--', linewidth=2, label='within-subtype')
plt.axvline(between_cutoff, color='black', linestyle='--', linewidth=2, label='between-subtype')

plt.xlim(left=0)
plt.xlabel("TreeSort divergence value for reassorting NA")
plt.ylabel("count")
plt.grid(alpha=0.3)

plt.savefig("plots/rea_events_div_cutoffs.pdf", dpi=300, bbox_inches="tight")
plt.show()
